 ## Задача 1

 Проверяем программно, что код, дуальный к двоичному коду Хэмминга,
 действительно является симплексным:
 - строим проверочную матрицу `H` кода Хэмминга;
 - берём дуальный код с порождающей матрицей `G_dual = H`;
 - перечисляем все кодовые слова дуального кода;
 - убеждаемся, что все ненулевые слова имеют одинаковый вес `2^(r-1)`.

In [8]:
from itertools import product

 ### Вспомогательные функции

 Здесь определены:
 - вес Хэмминга;
 - вывод вектора в удобном виде;
 - перечисление всех кодовых слов по порождающей матрице;
 - построение проверочной матрицы кода Хэмминга.

In [9]:
def weight(vector):  # считаем вес Хэмминга
    return sum(vector)


def format_vector(vector):  # форматируем вектор
    return "".join(map(str, vector))


def all_codewords(generator_matrix):  # перечисляем все кодовые слова
    k = len(generator_matrix)  # параметры кода
    n = len(generator_matrix[0])
    words = []

    for message in product([0, 1], repeat=k):  # перебираем все возможные слова
        codeword = []
        for column in range(n):  # перебираем все столбцы порождающей матрицы
            value = (
                sum(
                    message[row] * generator_matrix[row][column] for row in range(k)
                )  # считаем значение для каждого столбца
                % 2
            )
            codeword.append(value)  # добавляем значение в слово
        words.append(tuple(codeword))

    return words  # возвращаем все кодовые слова


def hamming_parity_check_matrix(
    r,
):  # строим проверочную матрицу, столбцы которой являются все ненулевыми векторами длины r
    columns = []
    for number in range(1, 2**r):  # проходим по всем ненулевым векторам длины r
        column = tuple(
            (number >> bit) & 1 for bit in range(r - 1, -1, -1)
        )  # переводим число в двоичный вектор
        columns.append(column)  # добавляем вектор в список

    return [
        [column[row] for column in columns] for row in range(r)
    ]  # возвращаем матрицу


def print_matrix(name, matrix):  # выводим матрицу
    print(f"{name} =")
    for row in matrix:
        print(" ", row)

 ### Основная проверка

 Для заданного `r` строим код Хэмминга `(2^r - 1, 2^r - 1 - r)`,
 затем его дуальный код и проверяем, что все ненулевые слова имеют
 одинаковый вес `2^(r-1)`.

In [10]:
def analyze_dual_of_hamming(r): # проверяем, что все ненулевые слова имеют одинаковый вес
    n = 2**r - 1 # параметры кода
    k = n - r # параметры кода
    expected_weight = 2 ** (r - 1) # ожидаемый вес

    print("=" * 72)
    print(f"Код Хэмминга: ({n}, {k}), r = {r}")
    print("=" * 72)

    parity_check = hamming_parity_check_matrix(r) # строим проверочную матрицу
    print_matrix("H", parity_check) # выводим матрицу

    dual_generator = parity_check # берём дуальный код с порождающей матрицей H
    dual_codewords = all_codewords(dual_generator) # перечисляем все кодовые слова дуального кода
    nonzero_codewords = [word for word in dual_codewords if any(word)] # перебираем все кодовые слова и выбираем ненулевые
    nonzero_weights = [weight(word) for word in nonzero_codewords] # считаем вес каждого ненулевого слова

    print(f"\nПараметры дуального кода: ({n}, {r})")
    print(f"Число кодовых слов: 2^{r} = {len(dual_codewords)}")
    print(
        f"Ожидаемый вес ненулевых слов для симплексного кода: 2^({r}-1) = {expected_weight}"
    )

    print("\nВсе кодовые слова дуального кода:")
    for index, word in enumerate(dual_codewords):
        print(f"{index:2d}: {format_vector(word)}   wt = {weight(word)}")

    unique_nonzero_weights = sorted(set(nonzero_weights)) # сортируем и убираем дубликаты
    is_simplex = unique_nonzero_weights == [expected_weight] # проверяем, что все ненулевые слова имеют одинаковый вес

    print("\nВеса ненулевых кодовых слов:", unique_nonzero_weights)
    print("Все ненулевые слова имеют одинаковый вес:", is_simplex)

    if is_simplex:
        print(
            f"Вывод: дуальный к коду Хэмминга ({n}, {k}) является симплексным кодом, "
            f"так как все его ненулевые слова имеют вес {expected_weight}."
        )
    else:
        print("Вывод: проверка не прошла, код не является симплексным.")

    print()

 ### Проверка на примерах

 В условии предлагается проверить хотя бы для `(7,4)` и `(15,11)`.

In [11]:
# Проверяем два примера из условия.
analyze_dual_of_hamming(3)  # (7,4) 
analyze_dual_of_hamming(4)  # (15,11)

Код Хэмминга: (7, 4), r = 3
H =
  [0, 0, 0, 1, 1, 1, 1]
  [0, 1, 1, 0, 0, 1, 1]
  [1, 0, 1, 0, 1, 0, 1]

Параметры дуального кода: (7, 3)
Число кодовых слов: 2^3 = 8
Ожидаемый вес ненулевых слов для симплексного кода: 2^(3-1) = 4

Все кодовые слова дуального кода:
 0: 0000000   wt = 0
 1: 1010101   wt = 4
 2: 0110011   wt = 4
 3: 1100110   wt = 4
 4: 0001111   wt = 4
 5: 1011010   wt = 4
 6: 0111100   wt = 4
 7: 1101001   wt = 4

Веса ненулевых кодовых слов: [4]
Все ненулевые слова имеют одинаковый вес: True
Вывод: дуальный к коду Хэмминга (7, 4) является симплексным кодом, так как все его ненулевые слова имеют вес 4.

Код Хэмминга: (15, 11), r = 4
H =
  [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
  [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1]
  [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1]
  [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]

Параметры дуального кода: (15, 4)
Число кодовых слов: 2^4 = 16
Ожидаемый вес ненулевых слов для симплексного кода: 2^(4-1) = 8

Все кодовые слова 